# Step 5 - Loss Ratio & Profitability Analysis

Notebook ini fokus menjawab: **segmen mana yang paling menguntungkan dan paling merugikan?**

Framework yang dipakai:
- Gross Loss Ratio = Gross Incurred Claim / Gross Written Premium
- Net Loss Ratio = Net Incurred Claim / Net Premium
- Ranking per segmen dengan filter kredibilitas
- Perbandingan gross vs net position setelah reasuransi
- Analisis large claim sebagai sensitivity analysis
- Analisis tren loss ratio per underwriting year
- Tambahan underwriting result sebagai indikator profitabilitas sederhana


## Ringkasan konsep bisnis

Dari diskusi bisnis sebelumnya:
- `GWP_IDR` = gross written premium, yaitu premi kotor yang dibayar nasabah.
- `RWP_IDR` = premi yang diberikan ke reasuransi.
- `GWC_IDR` = komisi yang dibayar AXA ke agen atau broker.
- `RWC_IDR` = komisi dari reasuransi.
- `GRS_ST_IDR` = klaim kotor yang sudah dibayar.
- `RI_ST_IDR` = bagian settled claim yang ditanggung reasuransi.
- `GRS_OS_IDR` = outstanding claim gross, yaitu cadangan klaim yang belum selesai.
- `RI_OS_IDR` = bagian outstanding claim yang diperkirakan ditanggung reasuransi.

Formula utama:
- Gross Claim = `GRS_ST_IDR + GRS_OS_IDR`
- Reinsurance Recovery = `RI_ST_IDR + RI_OS_IDR`
- Net Claim = `Gross Claim - Reinsurance Recovery`
- Net Premium = `GWP_IDR - RWP_IDR`
- Net Commission = `GWC_IDR - RWC_IDR`
- Gross Underwriting Result = `GWP_IDR - Gross Claim - GWC_IDR`
- Net Underwriting Result = `Net Premium - Net Claim - Net Commission`

Catatan penting interpretasi reasuransi:
- `Net Loss Ratio` bisa lebih tinggi dari `Gross Loss Ratio` karena **net premium** lebih kecil dari gross premium.
- Karena itu, efektivitas reasuransi tidak aman dinilai hanya dari selisih `Gross LR - Net LR`.
- Untuk melihat bantuan reasuransi, kita tampilkan juga **reinsurance recovery** dan **claim relief ratio**.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', palette='viridis')

# Config yang mudah diubah di Colab
USE_COLAB_UPLOAD = True
LOCAL_FILE_PATH = '/content/Study Case for Univ Airlangga 2026 _Actuarial AXA (Sent 2026.05.26) (1).xlsx'

SEGMENT_COLS = ['COB', 'BRANCH_', 'CHANNEL_', 'PRODUCT_NAME']
FOCUS_SEGMENT = 'COB'
TOP_N = 10

# Filter kredibilitas supaya ranking tidak didominasi segmen kecil/outlier.
MIN_GWP_BY_SEGMENT = 1_000_000_000
MIN_NET_PREMIUM_BY_SEGMENT = 1_000_000_000
MIN_POLICY_COUNT = 50
MIN_CLAIM_POLICY_COUNT = 5

# Untuk product, pakai guardrail lebih ketat karena paling rawan outlier.
PRODUCT_MIN_POLICY_COUNT = 200
PRODUCT_MIN_CLAIM_POLICY_COUNT = 10
PRODUCT_MIN_NET_PREMIUM = 5_000_000_000
PRODUCT_MAX_NET_LR_FOR_MAIN_RANKING = 500

# Large claim didefinisikan dari gross claim di level policy.
LARGE_CLAIM_QUANTILE = 0.95

if USE_COLAB_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    file_path = next(iter(uploaded.keys()))
else:
    file_path = LOCAL_FILE_PATH

print(f'File yang dipakai: {file_path}')


In [ ]:
premium_cols = [
    'COB', 'BRANCH_', 'CHANNEL_', 'PRODUCT_NAME', 'POLICY_NO',
    'DT_INC', 'DT_END', 'GWP_IDR', 'RWP_IDR', 'GWC_IDR', 'RWC_IDR', 'SUM_INSURED'
]
claim_cols = [
    'COB', 'BRANCH_', 'CHANNEL_', 'POLICY_NO', 'CLM_REF', 'DT_CLM',
    'GRS_ST_IDR', 'RI_ST_IDR', 'GRS_OS_IDR', 'RI_OS_IDR'
]

premium = pd.read_excel(file_path, sheet_name='Raw Premium', usecols=premium_cols)
claim = pd.read_excel(file_path, sheet_name='Raw Claim', usecols=claim_cols)

for col in ['DT_INC', 'DT_END']:
    premium[col] = pd.to_datetime(premium[col], errors='coerce')
for col in ['DT_CLM']:
    claim[col] = pd.to_datetime(claim[col], errors='coerce')

for col in ['GWP_IDR', 'RWP_IDR', 'GWC_IDR', 'RWC_IDR', 'SUM_INSURED']:
    premium[col] = pd.to_numeric(premium[col], errors='coerce').fillna(0)
for col in ['GRS_ST_IDR', 'RI_ST_IDR', 'GRS_OS_IDR', 'RI_OS_IDR']:
    claim[col] = pd.to_numeric(claim[col], errors='coerce').fillna(0)

print('Premium shape :', premium.shape)
print('Claim shape   :', claim.shape)
display(premium.head())
display(claim.head())


In [ ]:
# Policy-level summary untuk menghindari double count saat join.
premium_policy = (
    premium.sort_values(['POLICY_NO', 'DT_INC'])
    .groupby('POLICY_NO', as_index=False)
    .agg({
        'COB': 'first',
        'BRANCH_': 'first',
        'CHANNEL_': 'first',
        'PRODUCT_NAME': 'first',
        'DT_INC': 'min',
        'DT_END': 'max',
        'GWP_IDR': 'sum',
        'RWP_IDR': 'sum',
        'GWC_IDR': 'sum',
        'RWC_IDR': 'sum',
        'SUM_INSURED': 'max'
    })
)

claim_policy = (
    claim.groupby('POLICY_NO', as_index=False)
    .agg({
        'CLM_REF': 'nunique',
        'DT_CLM': 'min',
        'GRS_ST_IDR': 'sum',
        'RI_ST_IDR': 'sum',
        'GRS_OS_IDR': 'sum',
        'RI_OS_IDR': 'sum'
    })
    .rename(columns={'CLM_REF': 'claim_count'})
)

claim_range_check = claim.merge(
    premium_policy[['POLICY_NO', 'DT_INC', 'DT_END']],
    on='POLICY_NO', how='left'
)

df = premium_policy.merge(claim_policy, on='POLICY_NO', how='left')
fill_cols = ['claim_count', 'GRS_ST_IDR', 'RI_ST_IDR', 'GRS_OS_IDR', 'RI_OS_IDR']
df[fill_cols] = df[fill_cols].fillna(0)

df['uw_year'] = df['DT_INC'].dt.year
df['net_premium'] = df['GWP_IDR'] - df['RWP_IDR']
df['gross_claim'] = df['GRS_ST_IDR'] + df['GRS_OS_IDR']
df['reinsurance_recovery'] = df['RI_ST_IDR'] + df['RI_OS_IDR']
df['net_claim'] = df['gross_claim'] - df['reinsurance_recovery']
df['net_commission'] = df['GWC_IDR'] - df['RWC_IDR']
df['gross_underwriting_result'] = df['GWP_IDR'] - df['gross_claim'] - df['GWC_IDR']
df['net_underwriting_result'] = df['net_premium'] - df['net_claim'] - df['net_commission']
df['has_claim'] = df['gross_claim'] > 0

positive_claims = df.loc[df['gross_claim'] > 0, 'gross_claim']
large_claim_threshold = positive_claims.quantile(LARGE_CLAIM_QUANTILE)
df['is_large_claim'] = df['gross_claim'] >= large_claim_threshold

qa_summary = pd.DataFrame({
    'metric': [
        'policy_count', 'claim_policy_count', 'gwp_total', 'net_premium_total',
        'gross_claim_total', 'reinsurance_recovery_total', 'net_claim_total',
        'gross_underwriting_result_total', 'net_underwriting_result_total', 'large_claim_threshold'
    ],
    'value': [
        len(df),
        int(df['has_claim'].sum()),
        df['GWP_IDR'].sum(),
        df['net_premium'].sum(),
        df['gross_claim'].sum(),
        df['reinsurance_recovery'].sum(),
        df['net_claim'].sum(),
        df['gross_underwriting_result'].sum(),
        df['net_underwriting_result'].sum(),
        large_claim_threshold
    ]
})

data_quality = pd.DataFrame({
    'check': [
        'duplicate rows in premium by policy',
        'claim rows before inception',
        'claim rows after policy end',
        'non-positive net premium policy count'
    ],
    'count': [
        int((premium.groupby('POLICY_NO').size() > 1).sum()),
        int(((claim_range_check['DT_CLM'].notna()) & (claim_range_check['DT_CLM'] < claim_range_check['DT_INC'])).sum()),
        int(((claim_range_check['DT_CLM'].notna()) & (claim_range_check['DT_CLM'] > claim_range_check['DT_END'])).sum()),
        int((df['net_premium'] <= 0).sum())
    ]
})

display(qa_summary)
display(data_quality)
display(df.head())


In [ ]:
def safe_ratio(numerator, denominator):
    return np.where(denominator != 0, numerator / denominator, np.nan)


def segment_filter(summary, group_col):
    base_filter = (
        (summary['gwp'] >= MIN_GWP_BY_SEGMENT) &
        (summary['net_premium'] >= MIN_NET_PREMIUM_BY_SEGMENT) &
        (summary['policy_count'] >= MIN_POLICY_COUNT) &
        (summary['claim_policy_count'] >= MIN_CLAIM_POLICY_COUNT)
    )

    if group_col == 'PRODUCT_NAME':
        base_filter = (
            (summary['gwp'] >= MIN_GWP_BY_SEGMENT) &
            (summary['net_premium'] >= PRODUCT_MIN_NET_PREMIUM) &
            (summary['policy_count'] >= PRODUCT_MIN_POLICY_COUNT) &
            (summary['claim_policy_count'] >= PRODUCT_MIN_CLAIM_POLICY_COUNT) &
            (summary['net_loss_ratio_pct'] <= PRODUCT_MAX_NET_LR_FOR_MAIN_RANKING)
        )

    return summary[base_filter].copy()


def summarize_segment(data, group_col):
    summary = (
        data.groupby(group_col, dropna=False)
        .agg(
            policy_count=('POLICY_NO', 'nunique'),
            claim_policy_count=('has_claim', 'sum'),
            large_claim_policy_count=('is_large_claim', 'sum'),
            gwp=('GWP_IDR', 'sum'),
            rwp=('RWP_IDR', 'sum'),
            net_premium=('net_premium', 'sum'),
            gross_claim=('gross_claim', 'sum'),
            reinsurance_recovery=('reinsurance_recovery', 'sum'),
            net_claim=('net_claim', 'sum'),
            gross_commission=('GWC_IDR', 'sum'),
            reinsurance_commission=('RWC_IDR', 'sum'),
            net_commission=('net_commission', 'sum'),
            gross_underwriting_result=('gross_underwriting_result', 'sum'),
            net_underwriting_result=('net_underwriting_result', 'sum')
        )
        .reset_index()
    )

    summary['gross_loss_ratio_pct'] = 100 * safe_ratio(summary['gross_claim'], summary['gwp'])
    summary['net_loss_ratio_pct'] = 100 * safe_ratio(summary['net_claim'], summary['net_premium'])
    summary['premium_ceded_pct'] = 100 * safe_ratio(summary['rwp'], summary['gwp'])
    summary['claim_relief_pct_of_gross_claim'] = 100 * safe_ratio(summary['reinsurance_recovery'], summary['gross_claim'])
    summary['net_vs_gross_lr_gap_pct_point'] = summary['net_loss_ratio_pct'] - summary['gross_loss_ratio_pct']
    summary['gross_underwriting_margin_pct'] = 100 * safe_ratio(summary['gross_underwriting_result'], summary['gwp'])
    summary['net_underwriting_margin_pct'] = 100 * safe_ratio(summary['net_underwriting_result'], summary['net_premium'])
    summary['profitability_flag'] = np.where(summary['net_loss_ratio_pct'] > 100, 'Loss-making', 'Profitable/acceptable')

    summary = segment_filter(summary, group_col)
    return summary.sort_values(['net_loss_ratio_pct', 'net_underwriting_result'], ascending=[True, False]).reset_index(drop=True)


def overall_metrics(data, scenario_name):
    total_gwp = data['GWP_IDR'].sum()
    total_rwp = data['RWP_IDR'].sum()
    total_net_premium = data['net_premium'].sum()
    total_gross_claim = data['gross_claim'].sum()
    total_recovery = data['reinsurance_recovery'].sum()
    total_net_claim = data['net_claim'].sum()
    total_gross_uw = data['gross_underwriting_result'].sum()
    total_net_uw = data['net_underwriting_result'].sum()

    return pd.DataFrame({
        'scenario': [scenario_name],
        'policy_count': [data['POLICY_NO'].nunique()],
        'claim_policy_count': [int(data['has_claim'].sum())],
        'gwp': [total_gwp],
        'rwp': [total_rwp],
        'net_premium': [total_net_premium],
        'gross_claim': [total_gross_claim],
        'reinsurance_recovery': [total_recovery],
        'net_claim': [total_net_claim],
        'gross_loss_ratio_pct': [100 * total_gross_claim / total_gwp],
        'net_loss_ratio_pct': [100 * total_net_claim / total_net_premium],
        'premium_ceded_pct': [100 * total_rwp / total_gwp],
        'claim_relief_pct_of_gross_claim': [100 * total_recovery / total_gross_claim if total_gross_claim != 0 else np.nan],
        'gross_underwriting_result': [total_gross_uw],
        'net_underwriting_result': [total_net_uw]
    })


segment_summaries = {col: summarize_segment(df, col) for col in SEGMENT_COLS}

# Outlier products dipisahkan dari ranking utama.
product_all = (
    df.groupby('PRODUCT_NAME', dropna=False)
    .agg(
        policy_count=('POLICY_NO', 'nunique'),
        claim_policy_count=('has_claim', 'sum'),
        net_premium=('net_premium', 'sum'),
        net_claim=('net_claim', 'sum')
    )
    .reset_index()
)
product_all['net_loss_ratio_pct'] = 100 * safe_ratio(product_all['net_claim'], product_all['net_premium'])
product_outliers = product_all[
    (product_all['policy_count'] >= PRODUCT_MIN_POLICY_COUNT) &
    (product_all['claim_policy_count'] >= PRODUCT_MIN_CLAIM_POLICY_COUNT) &
    (product_all['net_premium'] >= PRODUCT_MIN_NET_PREMIUM) &
    (product_all['net_loss_ratio_pct'] > PRODUCT_MAX_NET_LR_FOR_MAIN_RANKING)
].sort_values('net_loss_ratio_pct', ascending=False)

for col, summary in segment_summaries.items():
    print(f'=== Best segments by {col} ===')
    display(summary.head(TOP_N))
    print(f'=== Worst segments by {col} ===')
    display(summary.sort_values(['net_loss_ratio_pct', 'net_underwriting_result'], ascending=[False, True]).head(TOP_N))

print('=== Product outliers excluded from main ranking ===')
display(product_outliers.head(TOP_N))


In [ ]:
focus_summary = segment_summaries[FOCUS_SEGMENT].copy()
display(focus_summary)

chart_data = focus_summary.melt(
    id_vars=[FOCUS_SEGMENT],
    value_vars=['gross_loss_ratio_pct', 'net_loss_ratio_pct'],
    var_name='ratio_type',
    value_name='loss_ratio_pct'
)

plt.figure(figsize=(14, 6))
sns.barplot(data=chart_data, x=FOCUS_SEGMENT, y='loss_ratio_pct', hue='ratio_type', palette='viridis')
plt.title(f'Gross vs Net Loss Ratio by {FOCUS_SEGMENT}')
plt.xlabel(FOCUS_SEGMENT)
plt.ylabel('Loss Ratio (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(
    data=focus_summary.sort_values('claim_relief_pct_of_gross_claim', ascending=False),
    x=FOCUS_SEGMENT, y='claim_relief_pct_of_gross_claim', palette='viridis'
)
plt.title(f'Reinsurance Claim Relief by {FOCUS_SEGMENT}')
plt.xlabel(FOCUS_SEGMENT)
plt.ylabel('Claim relief (% of gross claim)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

best_focus = focus_summary.nsmallest(1, 'net_loss_ratio_pct')
worst_focus = focus_summary.nlargest(1, 'net_loss_ratio_pct')

print('Segmen paling menguntungkan pada dimensi fokus:')
display(best_focus[[FOCUS_SEGMENT, 'policy_count', 'claim_policy_count', 'gross_loss_ratio_pct', 'net_loss_ratio_pct', 'net_underwriting_result']])
print('Segmen paling merugikan pada dimensi fokus:')
display(worst_focus[[FOCUS_SEGMENT, 'policy_count', 'claim_policy_count', 'gross_loss_ratio_pct', 'net_loss_ratio_pct', 'net_underwriting_result']])


In [ ]:
# Large claim analysis ini bersifat sensitivity analysis, bukan proyeksi realistis.
# Premium tetap dipertahankan, hanya impact claim besar yang di-zero-out.
df_no_large = df.copy()
df_no_large.loc[df_no_large['is_large_claim'], ['gross_claim', 'reinsurance_recovery', 'net_claim']] = 0
df_no_large['gross_underwriting_result'] = df_no_large['GWP_IDR'] - df_no_large['gross_claim'] - df_no_large['GWC_IDR']
df_no_large['net_underwriting_result'] = df_no_large['net_premium'] - df_no_large['net_claim'] - df_no_large['net_commission']
df_no_large['has_claim'] = df_no_large['gross_claim'] > 0

portfolio_compare = pd.concat([
    overall_metrics(df, 'Actual portfolio'),
    overall_metrics(df_no_large, 'Sensitivity: excluding large-claim impact')
], ignore_index=True)
display(portfolio_compare)

impact_rows = []
for seg_value in focus_summary[FOCUS_SEGMENT].tolist():
    base = df[df[FOCUS_SEGMENT] == seg_value].copy()
    adj = df_no_large[df_no_large[FOCUS_SEGMENT] == seg_value].copy()
    base_metrics = overall_metrics(base, 'base').iloc[0]
    adj_metrics = overall_metrics(adj, 'adjusted').iloc[0]
    impact_rows.append({
        FOCUS_SEGMENT: seg_value,
        'net_lr_actual_pct': base_metrics['net_loss_ratio_pct'],
        'net_lr_sensitivity_ex_large_pct': adj_metrics['net_loss_ratio_pct'],
        'delta_pct_point': base_metrics['net_loss_ratio_pct'] - adj_metrics['net_loss_ratio_pct'],
        'net_uw_actual': base_metrics['net_underwriting_result'],
        'net_uw_sensitivity_ex_large': adj_metrics['net_underwriting_result']
    })

impact_by_focus = pd.DataFrame(impact_rows).sort_values('delta_pct_point', ascending=False)
display(impact_by_focus)

plt.figure(figsize=(14, 6))
sns.barplot(data=impact_by_focus, x=FOCUS_SEGMENT, y='delta_pct_point', palette='viridis')
plt.title(f'Sensitivity to Large Claims - Reduction in Net LR by {FOCUS_SEGMENT}')
plt.xlabel(FOCUS_SEGMENT)
plt.ylabel('Reduction in Net LR (pct point)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
yearly = (
    df.groupby('uw_year', as_index=False)
    .agg(
        gwp=('GWP_IDR', 'sum'),
        rwp=('RWP_IDR', 'sum'),
        net_premium=('net_premium', 'sum'),
        gross_claim=('gross_claim', 'sum'),
        reinsurance_recovery=('reinsurance_recovery', 'sum'),
        net_claim=('net_claim', 'sum'),
        gross_underwriting_result=('gross_underwriting_result', 'sum'),
        net_underwriting_result=('net_underwriting_result', 'sum')
    )
)

yearly['gross_loss_ratio_pct'] = 100 * yearly['gross_claim'] / yearly['gwp']
yearly['net_loss_ratio_pct'] = 100 * yearly['net_claim'] / yearly['net_premium']
yearly['premium_ceded_pct'] = 100 * yearly['rwp'] / yearly['gwp']
yearly['claim_relief_pct_of_gross_claim'] = 100 * safe_ratio(yearly['reinsurance_recovery'], yearly['gross_claim'])
display(yearly)

plt.figure(figsize=(12, 5))
sns.lineplot(data=yearly, x='uw_year', y='gross_loss_ratio_pct', marker='o', label='Gross LR', color=plt.cm.viridis(0.2))
sns.lineplot(data=yearly, x='uw_year', y='net_loss_ratio_pct', marker='o', label='Net LR', color=plt.cm.viridis(0.8))
plt.title('Loss Ratio Trend by Underwriting Year')
plt.xlabel('Underwriting Year')
plt.ylabel('Loss Ratio (%)')
plt.tight_layout()
plt.show()

selected_segments = pd.concat([
    focus_summary.nsmallest(3, 'net_loss_ratio_pct')[[FOCUS_SEGMENT]],
    focus_summary.nlargest(3, 'net_loss_ratio_pct')[[FOCUS_SEGMENT]]
])[FOCUS_SEGMENT].drop_duplicates().tolist()

trend_by_segment = (
    df[df[FOCUS_SEGMENT].isin(selected_segments)]
    .groupby(['uw_year', FOCUS_SEGMENT], as_index=False)
    .agg(
        net_premium=('net_premium', 'sum'),
        net_claim=('net_claim', 'sum')
    )
)
trend_by_segment['net_loss_ratio_pct'] = 100 * trend_by_segment['net_claim'] / trend_by_segment['net_premium']
display(trend_by_segment)

plt.figure(figsize=(14, 6))
sns.lineplot(data=trend_by_segment, x='uw_year', y='net_loss_ratio_pct', hue=FOCUS_SEGMENT, marker='o', palette='viridis')
plt.title(f'Net Loss Ratio Trend by Underwriting Year - {FOCUS_SEGMENT}')
plt.xlabel('Underwriting Year')
plt.ylabel('Net Loss Ratio (%)')
plt.tight_layout()
plt.show()


In [ ]:
print('INTERPRETATION GUIDE')
print('- Net Loss Ratio > 100% => segmen loss-making dari sudut pandang klaim terhadap net premium.')
print('- Net Underwriting Result < 0 => profit underwriting sederhana negatif setelah premi, klaim, dan komisi.')
print('- Reinsurance recovery tinggi => reasuransi membantu menurunkan beban claim secara nominal.')
print('- Net LR lebih tinggi dari Gross LR tidak otomatis berarti reasuransi buruk, karena net premium memang lebih kecil.')
print('- Large-claim section adalah sensitivity analysis ekstrem, bukan forecast realistis kondisi actual tanpa large claim.')
print('- Product ranking utama sudah mengecualikan outlier ekstrem; lihat tabel outlier secara terpisah.')

for col, summary in segment_summaries.items():
    best = summary.nsmallest(1, 'net_loss_ratio_pct')
    worst = summary.nlargest(1, 'net_loss_ratio_pct')
    print(f'\nDimensi: {col}')
    print('Paling menguntungkan:')
    display(best[[col, 'policy_count', 'claim_policy_count', 'gross_loss_ratio_pct', 'net_loss_ratio_pct', 'claim_relief_pct_of_gross_claim', 'net_underwriting_result', 'profitability_flag']])
    print('Paling merugikan:')
    display(worst[[col, 'policy_count', 'claim_policy_count', 'gross_loss_ratio_pct', 'net_loss_ratio_pct', 'claim_relief_pct_of_gross_claim', 'net_underwriting_result', 'profitability_flag']])

print('\nProduct outliers yang dikeluarkan dari ranking utama:')
display(product_outliers.head(TOP_N))
